# Cache Warmup

학습 전에 Drive 캐시를 미리 채워두는 노트북.
느린 두 단계를 분리해서 사전에 실행해두면, 본 학습 노트북에서 바로 넘어간다.

## 실행 순서
1. Step 1 — 환경 준비 (Drive 마운트, 패키지 설치)
2. Step 2 — Hard Negative 증강 캐시
3. Step 3 — Synthetic Positive 캐시

## 재실행 정책
- **소스 변경 없음**: 두 캐시 모두 자동으로 재사용 (빠름)
- **hard_negatives/ 이미지 추가/삭제**: Step 2만 재실행
- **sample_img/ 또는 backgrounds/ 변경**: Step 3만 재실행

In [ ]:
# Step 1: 환경 준비
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

%cd /content/drive/MyDrive/0422
!pip install -q ultralytics pillow

In [ ]:
# Step 2: Hard Negative 증강 캐시
# hard_negatives/ 이미지가 추가/삭제/교체된 경우에만 캐시가 자동 재생성된다.
# --crops-per-image: 이미지당 생성할 정사각형 랜덤 크롭 수

!python augment_hard_negatives_cache.py \
  --hard-negatives-dir /content/drive/MyDrive/hard_negatives \
  --cache-dir /content/drive/MyDrive/hard_negative_cache_aug \
  --crops-per-image 1

In [ ]:
# Step 3: Synthetic Positive 캐시
# sample_img/ 또는 backgrounds/ 변경 시 자동 재생성된다.
# --cache-only: dataset 생성 없이 캐시만 채우고 종료
# --synthetic-cache-size: 캐시에 저장할 총 이미지 수 (학습마다 이 중 --synthetic-target 장 무작위 추출)
# --background-augment-copies: 배경당 증강 변형본 수 (0이면 원본만 사용)

!python build_real_prototype_dataset.py \
  --real-root /content/drive/MyDrive \
  --cache-only \
  --synthetic-cache-dir /content/drive/MyDrive/synthetic_cache_bgaug \
  --synthetic-cache-size 1000 \
  --background-augment-copies 3